In [ ]:
  import os
  ROOT = "/teamspace/studios/this_studio/internship"
  if not os.path.exists(ROOT):
      !git clone https://github.com/shaunmarv3/internship.git {ROOT}
  %cd {ROOT}
  !git pull --ff-only                                  # commit 2f5bf49 (band-2 loader fix)
  !pip -q install -r requirements.txt
  !pip -q install -U huggingface_hub
  !sudo apt-get update -qq && sudo apt-get install -y -qq p7zip-full   # fast multithreaded 7z


In [ ]:
  import os, shutil, subprocess
  from pathlib import Path
  import rasterio
  from huggingface_hub import hf_hub_download

  REPO_ID = "shaunmarvell/maritime-oil-data"
  ROOT = Path("/teamspace/studios/this_studio/internship"); os.chdir(ROOT)

  IMG,  MSK  = ROOT/"data/oil/images",      ROOT/"data/oil/masks"        # Part I + II (train/val)
  TIMG, TMSK = ROOT/"data/oil_test/images", ROOT/"data/oil_test/masks"   # Part III (held-out test)
  for d in (IMG, MSK, TIMG, TMSK): d.mkdir(parents=True, exist_ok=True)
  STG = ROOT/"stg"; STG.mkdir(exist_ok=True)

  RASTER = {".tif", ".tiff", ".png"}
  def bandcount(p):
      try:
          with rasterio.open(p) as s: return s.count
      except Exception: return -1

  def organize(src, dst_img, dst_msk, prefix, kind):
      """Move rasters into images/masks with a stem prefix (collision-safe pairing)."""
      ni = nm = 0
      for p in Path(src).rglob("*"):
          if not p.is_file() or p.suffix.lower() not in RASTER: continue
          k = kind if kind in ("image", "mask") else ("image" if bandcount(p) == 2 else "mask")
          dst = (Path(dst_img) if k == "image" else Path(dst_msk)) / f"{prefix}{p.stem}{p.suffix.lower()}"
          shutil.move(str(p), str(dst))
          ni += (k == "image"); nm += (k == "mask")
      print(f"  {prefix} -> {ni} images, {nm} masks")

  # (archive, stem-prefix, forced-kind|'band', dest_images, dest_masks)
  JOBS = [
      ("part1/01_Train_Val_Oil_Spill_images.7z", "p1_",  "image", IMG,  MSK),
      ("part1/01_Train_Val_Oil_Spill_mask.7z",   "p1_",  "mask",  IMG,  MSK),
      ("part2/01_Train_Val_Lookalike_images.7z", "p2l_", "image", IMG,  MSK),
      ("part2/01_Train_Val_Lookalike_mask.7z",   "p2l_", "mask",  IMG,  MSK),
      ("part2/01_Train_Val_No_Oil_Images.7z",    "p2n_", "image", IMG,  MSK),
      ("part2/01_Train_Val_No_Oil_mask.7z",      "p2n_", "mask",  IMG,  MSK),
      ("part3/02_Test_images_and_ground_truth.7z","p3_", "band",  TIMG, TMSK),  # mixed -> split by band count
  ]

  for rel, prefix, kind, di, dm in JOBS:
      print("===", rel)
      local = hf_hub_download(REPO_ID, rel, repo_type="dataset", local_dir=str(ROOT/"hf_dl"))
      ex = STG/Path(rel).stem
      if ex.exists(): shutil.rmtree(ex)
      ex.mkdir(parents=True)
      subprocess.run(["7z", "x", local, f"-o{ex}", "-y", "-mmt=on", "-bso0", "-bsp0"], check=True)
      organize(ex, di, dm, prefix, kind)
      shutil.rmtree(ex); os.remove(local)              # free staging + archive (keeps disk low)
      print("   cleaned staging + archive")

  print("\nTRAINVAL  images:", len(list(IMG.glob('*'))),  " masks:", len(list(MSK.glob('*'))))
  print("TEST      images:", len(list(TIMG.glob('*'))), " masks:", len(list(TMSK.glob('*'))))


In [ ]:
import os, shutil, subprocess, time
from pathlib import Path
import rasterio
from huggingface_hub import hf_hub_download, login

login(os.environ["HF_TOKEN"])          # <-- set HF_TOKEN in the environment (lifts the rate limit)

REPO_ID = "shaunmarvell/maritime-oil-data"
ROOT = Path("/teamspace/studios/this_studio/internship"); os.chdir(ROOT)
IMG, MSK   = ROOT/"data/oil/images",      ROOT/"data/oil/masks"
TIMG, TMSK = ROOT/"data/oil_test/images", ROOT/"data/oil_test/masks"
STG = ROOT/"stg"; STG.mkdir(exist_ok=True)
RASTER = {".tif", ".tiff", ".png"}
def bandcount(p):
    try:
        with rasterio.open(p) as s: return s.count
    except Exception: return -1
def organize(src, di, dm, prefix, kind):
    ni = nm = 0
    for p in Path(src).rglob("*"):
        if not p.is_file() or p.suffix.lower() not in RASTER: continue
        k = kind if kind in ("image","mask") else ("image" if bandcount(p)==2 else "mask")
        dst = (Path(di) if k=="image" else Path(dm))/f"{prefix}{p.stem}{p.suffix.lower()}"
        shutil.move(str(p), str(dst)); ni += k=="image"; nm += k=="mask"
    return ni, nm

# ONLY the remaining archives (p1 + lookalike already done = 1885 pairs)
JOBS = [
    ("part2/01_Train_Val_No_Oil_Images.7z", "p2n_", "image", IMG,  MSK),
    ("part2/01_Train_Val_No_Oil_mask.7z",   "p2n_", "mask",  IMG,  MSK),
    ("part3/02_Test_images_and_ground_truth.7z","p3_","band", TIMG, TMSK),
]
for i, (rel, prefix, kind, di, dm) in enumerate(JOBS, 1):
    t0 = time.time(); print(f"\n[{i}/{len(JOBS)}] {rel}  ↓ downloading...", flush=True)
    local = hf_hub_download(REPO_ID, rel, repo_type="dataset", local_dir=str(ROOT/"hf_dl"))
    print(f"   ✓ {os.path.getsize(local)/1e9:.1f} GB in {time.time()-t0:.0f}s — extracting...", flush=True)
    ex = STG/Path(rel).stem
    if ex.exists(): shutil.rmtree(ex)
    ex.mkdir(parents=True)
    subprocess.run(["7z", "x", local, f"-o{ex}", "-y", "-mmt=on", "-bsp1"], check=True)
    ni, nm = organize(ex, di, dm, prefix, kind)
    shutil.rmtree(ex); os.remove(local)
    print(f"   ✓ {prefix}: {ni} images, {nm} masks | {time.time()-t0:.0f}s", flush=True)

print("\n==== DONE ====")
print("TRAINVAL images:", len(list(IMG.glob('*'))), " masks:", len(list(MSK.glob('*'))))
print("TEST     images:", len(list(TIMG.glob('*'))), " masks:", len(list(TMSK.glob('*'))))

In [ ]:
  import os, shutil, subprocess
  from pathlib import Path
  import rasterio
  from huggingface_hub import hf_hub_download, login

  login("")   # re-auth (kernel may have restarted)
  REPO_ID = "shaunmarvell/maritime-oil-data"
  ROOT = Path("/teamspace/studios/this_studio/internship"); os.chdir(ROOT)
  TIMG, TMSK = ROOT/"data/oil_test/images", ROOT/"data/oil_test/masks"
  for d in (TIMG, TMSK):                       # clear stale partial test set
      if d.exists(): shutil.rmtree(d)
      d.mkdir(parents=True)
  STG = ROOT/"stg"; STG.mkdir(exist_ok=True)

  RASTER = {".tif", ".tiff", ".png"}
  def bandcount(p):
      try:
          with rasterio.open(p) as s: return s.count
      except Exception: return -1

  rel = "part3/02_Test_images_and_ground_truth.7z"
  print("↓ downloading Part III (resumes 5.3 GB partial)...", flush=True)
  local = hf_hub_download(REPO_ID, rel, repo_type="dataset", local_dir=str(ROOT/"hf_dl"))
  print(f"✓ {os.path.getsize(local)/1e9:.1f} GB — extracting...", flush=True)
  ex = STG/"part3"
  if ex.exists(): shutil.rmtree(ex)
  ex.mkdir(parents=True)
  subprocess.run(["7z", "x", local, f"-o{ex}", "-y", "-mmt=on", "-bsp1"], check=True)

  ni = nm = 0; samples = []
  for p in Path(ex).rglob("*"):
      if not p.is_file() or p.suffix.lower() not in RASTER: continue
      bc = bandcount(p)
      k = "image" if bc == 2 else "mask"
      if len(samples) < 12: samples.append((str(p.relative_to(ex)), bc))
      dst = (TIMG if k == "image" else TMSK)/f"p3_{p.stem}{p.suffix.lower()}"
      shutil.move(str(p), str(dst)); ni += k == "image"; nm += k == "mask"
  print("\nsample files (relpath, bandcount):")
  for s in samples: print("  ", s)
  shutil.rmtree(ex); os.remove(local)
  print(f"\nTEST images: {ni}   masks: {nm}")


In [ ]:
  import os, shutil, subprocess
  from pathlib import Path
  import rasterio
  from huggingface_hub import hf_hub_download, login

  login("")
  REPO_ID = "shaunmarvell/maritime-oil-data"
  ROOT = Path("/teamspace/studios/this_studio/internship"); os.chdir(ROOT)
  TIMG, TMSK = ROOT/"data/oil_test/images", ROOT/"data/oil_test/masks"
  for d in (TIMG, TMSK):                       # wipe the broken 150-file test set
      if d.exists(): shutil.rmtree(d)
      d.mkdir(parents=True)
  STG = ROOT/"stg"; STG.mkdir(exist_ok=True)

  RASTER = {".tif", ".tiff", ".png"}
  def bandcount(p):
      try:
          with rasterio.open(p) as s: return s.count
      except Exception: return -1
  def category(s):                              # from the file's path
      s = s.lower()
      if "look" in s: return "l"
      if "no" in s and "oil" in s: return "n"   # "No oil"
      if "oil" in s: return "o"
      return "x"

  rel = "part3/02_Test_images_and_ground_truth.7z"
  print("↓ downloading Part III...", flush=True)
  local = hf_hub_download(REPO_ID, rel, repo_type="dataset", local_dir=str(ROOT/"hf_dl"))
  print(f"✓ {os.path.getsize(local)/1e9:.1f} GB — extracting...", flush=True)
  ex = STG/"part3"
  if ex.exists(): shutil.rmtree(ex)
  ex.mkdir(parents=True)
  subprocess.run(["7z", "x", local, f"-o{ex}", "-y", "-mmt=on", "-bso0", "-bsp0"], check=True)

  ni = nm = 0; cats = {}
  for p in Path(ex).rglob("*"):
      if not p.is_file() or p.suffix.lower() not in RASTER: continue
      cat = category(str(p.relative_to(ex)))
      is_mask = (bandcount(p) == 1)
      stem = p.stem.replace("_segmentation", "")          # strip suffix so stems align
      name = f"p3{cat}_{stem}{p.suffix.lower()}"          # per-category prefix avoids collision
      shutil.move(str(p), str((TMSK if is_mask else TIMG)/name))
      if is_mask: nm += 1
      else: ni += 1
      cats[cat] = cats.get(cat, 0) + 1
  shutil.rmtree(ex); os.remove(local)

  imgs = {q.stem for q in TIMG.rglob('*') if q.is_file()}
  msks = {q.stem for q in TMSK.rglob('*') if q.is_file()}
  print(f"\nTEST images: {ni}  masks: {nm}  | per-category file counts: {cats}")
  print(f"MATCHED STEMS: {len(imgs & msks)}   (expect ~450)")
  print("img stems:", sorted(imgs)[:3], "| msk stems:", sorted(msks)[:3])


In [ ]:
  from pathlib import Path
  for tag, root in [("TRAIN", "data/oil"), ("TEST", "data/oil_test")]:
      imgs = {p.stem for p in Path(root, "images").rglob("*") if p.is_file()}
      msks = {p.stem for p in Path(root, "masks").rglob("*") if p.is_file()}
      print(f"\n{tag}: {len(imgs)} imgs, {len(msks)} masks, MATCHED STEMS: {len(imgs & msks)}")
      print("  img stems:", sorted(imgs)[:3])
      print("  msk stems:", sorted(msks)[:3])


In [ ]:
  !python src/data/inspect_oil_data.py --data_root data/oil      --label "TRAIN I+II" --n 8
  !python src/data/inspect_oil_data.py --data_root data/oil_test --label "TEST III"   --n 8

In [ ]:
  import torch; print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB")

In [ ]:
  %cd /teamspace/studios/this_studio/internship
  !git pull --ff-only      # gets commit 11874e3 (fast mask-only scan)


In [ ]:
  !python src/models/train_segmentation.py --model segformer --backbone b4 \
    --dataset_type zenodo --data_root data/oil \
    --img_size 512 --epochs 50 --batch_size 16 --lr 6e-5 \
    --num_workers 8 --checkpoint_dir checkpoints/oil --wandb_project maritime-oil-spill


In [ ]:
  !python src/models/train_segmentation.py --model segformer --backbone b4 \
    --dataset_type zenodo --data_root data/oil \
    --img_size 512 --epochs 50 --batch_size 16 --lr 6e-5 \
    --num_workers 8 --checkpoint_dir checkpoints/oil --no_wandb


In [ ]:
  import os
  os.environ["WANDB_API_KEY"] = ""   # from wandb.ai/authorize
  os.environ["HF_TOKEN"]      = ""     # your HF write token
  nproc = min(os.cpu_count(), 24); print("workers:", nproc, "of", os.cpu_count())

  for m in ["segformer", "deeplabv3+", "unet"]:
      print(f"\n########## {m} ##########\n")
      !python src/models/train_segmentation.py --model {m} --backbone b4 \
        --dataset_type zenodo --data_root data/oil \
        --img_size 512 --epochs 50 --batch_size 16 --lr 6e-5 \
        --num_workers {nproc} --checkpoint_dir checkpoints/oil \
        --wandb_project maritime-oil-spill --push_hf


In [ ]:
  import json, torch
  from pathlib import Path
  from torch.utils.data import DataLoader
  from src.data.loaders import OilSpillDataset, get_oil_transforms, OIL_CLASSES_BINARY
  from src.models.segmentation import build_segmentation_model, SegmentationMetrics

  # Cache test preprocessing ONCE (it's reused across all 3 models → no triple Lee-filter)
  test_ds = OilSpillDataset("data/oil_test", split="all", dataset_type="zenodo",
                            transform=get_oil_transforms(512, "val"), img_size=512,
                            val_split=0, cache_dir="data/oil_test/.preproc_cache")
  test_ld = DataLoader(test_ds, batch_size=16, num_workers=8, pin_memory=True)
  print(f"test pairs: {len(test_ds)}")        # expect 450

  rows = []
  print(f"\n{'model':12} {'mIoU':>8} {'OilIoU':>8} {'valOilIoU':>10}")
  for m, ck in [("deeplabv3+","best_deeplabv3+.pt"),
                ("segformer","best_segformer.pt"),
                ("unet","best_unet.pt")]:
      p = Path("checkpoints/oil")/ck
      if not p.exists():
          print(f"{m:12}   (skipped — {ck} not found yet)"); continue
      c  = torch.load(p, map_location="cuda")
      bb = c.get("args", {}).get("backbone", "b4")     # build with the SAME backbone it trained with
      model = build_segmentation_model(m, num_classes=2, backbone=bb).cuda()
      model.load_state_dict(c["model_state"]); model.eval()
      met = SegmentationMetrics(2, list(OIL_CLASSES_BINARY.values())); met.reset()
      with torch.no_grad():
          for x, y in test_ld:
              met.update(model(x.cuda()).float(), y.cuda())
      r = met.compute()
      oil = r.get("IoU_oil_spill", r.get("IoU_1", 0.0))
      val = c.get("oil_iou")
      rows.append({"model": m, "test_mIoU": round(r["mIoU"],4),
                   "test_OilIoU": round(oil,4),
                   "val_OilIoU": round(val,4) if isinstance(val,(int,float)) else None})
      print(f"{m:12} {r['mIoU']:8.4f} {oil:8.4f} {val if val else 0:10.4f}")

  Path("checkpoints/oil/test_comparison.json").write_text(json.dumps(rows, indent=2))
  print("\nsaved → checkpoints/oil/test_comparison.json")


In [ ]:
  !cd ~/internship  && git pull                 # get latest (SOS val fix + all M3 changes)

In [ ]:
  # ── 2. Verify old Zenodo data is there ────────────────────────────
  !echo "Zenodo train images: $(ls data/oil/images/*.tif 2>/dev/null | wc -l)"
  !echo "Zenodo test images:  $(ls data/oil_test/images/*.tif 2>/dev/null | wc -l)"


In [ ]:
  import os
  os.chdir("/home/zeus/content/internship")
  !git pull
  !echo "Zenodo train: $(ls data/oil/images/*.tif | wc -l) TIFs"
  !echo "Zenodo test:  $(ls data/oil_test/images/*.tif | wc -l) TIFs"


In [ ]:
  !python download_sos_data.py --dest data/sos


In [ ]:
  !python download_sos_data.py --dest data/sos
  !echo "SOS train: $(ls data/sos/images/train/*.png | wc -l) images"
  !echo "SOS val:   $(ls data/sos/images/val/*.png   | wc -l) images"

In [ ]:
  import os
  os.environ["WANDB_API_KEY"] = ""   # from wandb.ai/authorize
  os.environ["HF_TOKEN"]      = ""     # your HF write token
  nproc = min(os.cpu_count(), 24); print("workers:", nproc, "of", os.cpu_count())

In [ ]:
  !python src/models/train_segmentation.py \
      --model segformer --backbone b5 \
      --dataset_type zenodo --data_root data/oil \
      --epochs 50 --batch_size 16 --lr 4e-5 \
      --num_workers 8 --upsample_lookalike 2.0 \
      --wandb_project maritime-oil-spill --push_hf


In [ ]:
  !python src/models/eval_threshold.py \
      --checkpoint checkpoints/oil/best_segformer.pt \
      --test_data  data/oil_test